In [ ]:
import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../experiments'));

import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

import json

import importlib
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False
stiffness_pressure = 0.4
scale_factor_pressure = 0.01
avg_len = 0.2


In [ ]:
grid_size = 10
num_pipes = 20

In [ ]:
vertices, edges, boundary_vxs, boundary_lines = pattern_generator_using_gmsh.generate_voronoi_mesh(3, avg_len, avg_len, return_line_segments=True)


In [ ]:
# vertices = np.load("../experiments/parallelized_experiments/output/voronoi_5/2024_01_07_15_18//0/vertices_voronoi_5_0.npy")
# edges = np.load("../experiments/parallelized_experiments/output/voronoi_5/2024_01_07_15_18/0/edges_voronoi_5_0.npy")

In [ ]:
visualization.plot_line_segments(list(vertices) + list(boundary_vxs), list(edges - 1) + list(boundary_lines + len(vertices) - 1))

In [ ]:
import mesher_helper

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_from_embeddings_array_input_allow_boundary(avg_len,avg_len, None, boundary_vxs, boundary_lines, vertices, edges)
finalMarkers = np.where(np.array(fusing_data) == 1)[0]


In [ ]:
m = MeshFEM.Mesh(v, np.array(f) - 1)


In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=10, height=10)

In [ ]:
m, marker = pattern_generator_using_gmsh.generate_voronoi_mesh(5, avg_len, avg_len, shrink_percentage=0.5)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)



In [ ]:
# m, marker, dots = pattern_generator_using_gmsh.get_random_grid_pipes(avg_len, avg_len, grid_size, num_pipes)

# finalMarkers = np.where(np.array(marker) == 1)[0]
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

# fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

# ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)



In [ ]:
max(m.edgeLengths()), min(m.edgeLengths())

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=10, height=10)

In [ ]:
viewer = TriMeshViewer(ipu, width=1024, height=1024)
viewer.showWireframe(True)
viewer.show()

In [ ]:
np.max(la.norm(ipu.gradient(energyType = inflation.InflatablePeriodicUnit.EnergyType.Elastic)[3:-2].reshape(-1, 3), axis = 1))

In [ ]:
configure_solver_parallelism()

In [ ]:
viewer.update()
viewer.showWireframe(False)

In [ ]:
orender = viewer.offscreenRenderer(width=1024,height=1024)
# orender.meshes[0].setColor(C[mm.elements().ravel()])
orender.render()
orender.save("cosine_dash.png")

In [ ]:
viewer.getCameraParams()

In [ ]:
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
hessianShiftForRigidMotion = 1e-10
hessianShiftForAlphainPlanar = 1e-12

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

In [ ]:
fixedVars, hessianShift = [ipu.numVars() - 2], hessianShiftForRigidMotion

In [ ]:
opts.niter = 500
opts.gradTol = 1e-10

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)), hessianShiftForAlphainPlanar

In [ ]:
fixedVars

In [ ]:
opts.niter = 500
opts.gradTol = 1e-11

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
experiment_log = {}
experiment_log["Ipu simulation succeed"] = int(cr.success)
experiment_log["Simulation Kappa value"] = (ipu.getVars()[-2])
if np.abs(ipu.getVars()[-2]) > 1e-6:
    experiment_log["Planar equilibrium"] = 0
    print("Warning: Can not compute stiffness due to non-planar equilibrium!")
else:
    experiment_log["Planar equilibrium"] = 1

In [ ]:
ipu.getVars()[:3], ipu.getVars()[-2:]

In [ ]:
name = "voronoi"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
# time_stamp = "2023_10_25_14_54"
base_folder = 'output/{}/{}'.format(name, time_stamp)
    
result_folder = "{}/{}".format(base_folder, 0)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  
    
render_images = True
variable = 0

In [ ]:
stiffness_fixedVars = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 1, ipu.numVars() - 2]

In [ ]:
optimizer = inflation.get_inflation_optimizer(ipu, stiffness_fixedVars, opts, callback=cb, hessianShift = 0)

In [ ]:
prob = optimizer.get_problem()

In [ ]:
prob.hessian()

In [ ]:
import mode_viewer

import compute_vibrational_modes
class ModalAnalysisWrapper:
    def __init__(self, sheet):
        self.sheet = sheet
    def hessian(self):
        return self.sheet.hessian()

In [ ]:
# lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(prob), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=10, sigma=-1e-10, fixedVars = stiffness_fixedVars)


In [ ]:
# import mode_viewer, importlib
# mview = mode_viewer.ModeViewer(az_ipu, modes, lambdas, amplitude=200)
# mview.show()

In [ ]:
import periodic_simulation_setup, importlib
importlib.reload(periodic_simulation_setup)
from periodic_simulation_setup import *

In [ ]:
ipu.getBendingStiffnessFixedVars()

In [ ]:
ipu_stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(ipu, 1000, optimizer, hessianShift = 1e-7, fixedVars = ipu.getBendingStiffnessFixedVars()[3:], filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable), generate_images = render_images)

In [ ]:
min(ipu_stiffness_values), max(ipu_stiffness_values)

In [ ]:
min(ipu_stiffness_values), max(ipu_stiffness_values)

In [ ]:
(1.643545678892331 - 1.6431813417744152) / 1.6431813417744152

In [ ]:
min(ipu_stiffness_values), max(ipu_stiffness_values)

In [ ]:
(1.6438213444509293 - 1.6431813417744152) / 1.6431813417744152

In [ ]:
curr_vars = ipu.getVars()
curr_vars[-1] = 0.2
ipu.setVars(curr_vars)

In [ ]:
from IPython.display import Image
Image(filename="{}/stiffness_{}_{}.png".format(result_folder, name, variable)) 

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT, disableFusedRegionTFT)

In [ ]:
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)
az_viewer.show()

In [ ]:
framerate = 5 # Update every 5 iterations
def az_cb(it):
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
az_ipu.getVars()[az_ipu.get_average_z_idx()]

In [ ]:
az_ipu.getVars()[-2:]

In [ ]:
periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian())[-2:, -2:]

In [ ]:
vars = az_ipu.getVars()
vars[-2] = 0
vars[-1] += 0.1
az_ipu.setVars(vars)
print(az_ipu.gradient()[-2:])
periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian())[-2:, -2:]

In [ ]:
import fd_validation

In [ ]:

F_indices = [np.arange(0, 3)]
F_indices = np.array(F_indices).flatten()

u_indices = [np.arange(3, az_ipu.numVars() - 2)]
u_indices = np.array(u_indices).flatten()


R_star_indices = [np.arange(3 + az_ipu.ipu.numFluctuationDisplacementVars(), az_ipu.numVars())]
R_star_indices = np.array(R_star_indices).flatten()




In [ ]:

var_types = ['F', 'u', 'R']
var_indices = {'F': F_indices,
               'u': u_indices, 
               'R': R_star_indices}


In [ ]:
az_ipu.ipu.sheet.usingTensionFieldEnergy(False)

In [ ]:
# fd_validation.hessConvergencePlot(az_ipu.ipu.sheet)

In [ ]:
# fd_validation.hessian_convergence_block_plot(az_ipu, var_types, var_indices, customArgs={'energyType': ipu.EnergyType.Pressure})

In [ ]:
# if not allowBending:
#     fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
# else:
fixedVars, hessianShift = [az_ipu.get_average_z_idx(), periodic_unit_helper.get_center_fixedVars(ipu)[0], periodic_unit_helper.get_center_fixedVars(ipu)[1]], hessianShiftForAlphainPlanar
opts.niter = 1000
opts.gradTol = 1e-10
cr = inflation.inflation_newton(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)

In [ ]:
name = "mirror_cosine_dash"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
# time_stamp = "2023_10_25_14_54"
base_folder = 'output/{}/{}'.format(name, time_stamp)
    
result_folder = "{}/{}".format(base_folder, get_label(amp, r, angle))
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  
    
render_images = True
variable = get_label(amp, r, angle)

In [ ]:
stiffness_fixedVars = [az_ipu.get_average_z_idx(), periodic_unit_helper.get_center_fixedVars(ipu)[0], periodic_unit_helper.get_center_fixedVars(ipu)[1], az_ipu.numVars() - 1, az_ipu.numVars() - 2]

In [ ]:
az_optimizer = inflation.get_inflation_optimizer(az_ipu, stiffness_fixedVars, opts, callback=az_cb, hessianShift = 0)

In [ ]:
stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = 0, fixedVars = stiffness_fixedVars, filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable), generate_images = render_images)
np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
np.save("{}/stiffness_coefficient_{}_{}.npy".format(result_folder, name, variable), stiffness_coefficient)

In [ ]:
from IPython.display import Image
Image(filename="{}/stiffness_{}_{}.png".format(result_folder, name, variable)) 

In [ ]:
la.norm(stiffness_values - ipu_stiffness_values)

In [ ]:
np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(az_ipu.ipu))
np.save("{}/average_deformation_gradient_matrix_{}_{}.npy".format(result_folder, name, variable), get_deformation_matrix(az_ipu.ipu))

In [ ]:
min(stiffness_values), max(stiffness_values)

In [ ]:
import periodic_simulation_setup
import importlib
importlib.reload(periodic_simulation_setup)

In [ ]:
betas = np.linspace(0, 2 * np.pi, 1000)

In [ ]:
optimizer = inflation.get_inflation_optimizer(ipu, ipu.getStretchingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
stretchingStiffness = inflation.getStretchingStiffness(ipu, betas, optimizer, 0, ipu.getStretchingStiffnessFixedVars())

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(False)

In [ ]:
%%capture
ss_obj = stretching_stiffness_class(ipu, ipu.sheet, optimizer, viewer, 2)
ss_obj.setVars(ss_obj.getVars())

In [ ]:
fd_validation.gradConvergencePlot(ss_obj)

In [ ]:
fd_validation.secondDerivativeConvergencePlot(ss_obj,  epsilons = np.logspace(-6, -2, 20))

In [ ]:
stiffness_optimizer = inflation.get_inflation_optimizer(az_ipu, [az_ipu.get_average_z_idx(), periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[0], periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[1], az_ipu.numVars() - 2, az_ipu.numVars() - 1], opts, callback=cb, hessianShift = 0)

In [ ]:
bs_obj = periodic_simulation_setup.bending_stiffness_class(az_ipu, az_ipu.ipu.sheet, az_optimizer, az_viewer, fixedVars = [])

bs_obj.setVars(bs_obj.getVars())

fd_validation.secondDerivativeConvergencePlot(bs_obj, epsilons = np.logspace(-6, -1, 50))